# MCP：真实发现、调用与拒绝

Python 3.11+、CPU、无模型和密钥。使用官方 mcp 2.3.0、协议 2026-07-28；分别执行内存连接与独立 stdio 子进程。它不验证 HTTP/OAuth、网络重试或私有对象授权。

先预测：类型错误是否进入工具函数？负数业务错误呢？两种连接的结果相同，是否代表进程也相同？

In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','mcp==2.3.0'])


## 服务器与客户端文件

真实 handler 使用 ToolError 返回可处理业务错误。attempts 在函数入口增加，用于区分 schema 拒绝与业务拒绝。这里只做加法，不实施业务写入；status 的进程 ID 仅证明实验边界，不是用户身份。客户端显式检查 is_error，不能只读取 structured_content。

In [ ]:
from pathlib import Path
import json,subprocess,sys
Path("audit_server.py").write_text('import os\nfrom pydantic import BaseModel\nfrom mcp.server import MCPServer\nfrom mcp.server.mcpserver.exceptions import ToolError\nmcp = MCPServer(\'SuGuilin teaching calculator\')\nattempts = 0\nclass Status(BaseModel):\n    attempts: int\n    pid: int\n@mcp.tool()\ndef add(a: int, b: int) -> int:\n    """Add non-negative integers; this tool reads no user data."""\n    global attempts\n    attempts += 1\n    if a < 0 or b < 0:\n        raise ToolError(\'non-negative integers required\')\n    return a + b\n@mcp.tool()\ndef status() -> Status:\n    """Teaching instrumentation, not an authentication identity."""\n    return Status(attempts=attempts, pid=os.getpid())\nif __name__ == \'__main__\':\n    mcp.run(transport=\'stdio\')\n',encoding="utf-8")
Path("audit_client.py").write_text("import json,os,sys\nfrom pathlib import Path\nfrom importlib.metadata import version\nimport anyio\nfrom mcp import Client, StdioServerParameters\nfrom audit_server import mcp\nasync def run():\n    transport = mcp if sys.argv[1] == 'memory' else StdioServerParameters(\n        command=sys.executable, args=[str(Path('audit_server.py').resolve())])\n    async with Client(transport) as client:\n        assert client.protocol_version == '2026-07-28'\n        tools = await client.list_tools()\n        assert {'add','status'} == {t.name for t in tools.tools}\n        good = await client.call_tool('add', {'a':6,'b':7})\n        assert not good.is_error and good.structured_content == {'result':13}\n        business = await client.call_tool('add', {'a':-1,'b':7})\n        invalid = await client.call_tool('add', {'a':'not-a-number','b':7})\n        missing = await client.call_tool('missing', {})\n        assert business.is_error and invalid.is_error and missing.is_error\n        status = (await client.call_tool('status',{})).structured_content\n        assert status['attempts'] == 2  # valid and business-negative reached function\n        independent = status['pid'] != os.getpid()\n        assert independent == (sys.argv[1]=='stdio')\n        checks={'discovery':True,'structured_result':True,'invalid_input_rejected':True,\n                'business_error_retained':True,'unknown_tool_rejected':True,\n                'subprocess_boundary':independent if sys.argv[1]=='stdio' else True}\n        print(json.dumps({'mode':sys.argv[1], 'sdk':version('mcp'),\n            'protocol':client.protocol_version, 'checks':checks,\n            'sum':13, 'handler_attempts':status['attempts'],\n            'independent_process':independent}))\nif __name__=='__main__': anyio.run(run)\n",encoding="utf-8")


In [ ]:
results=[]
for mode in ['memory','stdio']:
    completed=subprocess.run([sys.executable,'audit_client.py',mode],capture_output=True,text=True,timeout=30)
    if completed.returncode: raise RuntimeError(completed.stderr)
    result=json.loads(completed.stdout.strip())
    results.append(result)
    print(json.dumps(result,ensure_ascii=False,indent=2))
assert all(all(r['checks'].values()) for r in results)


## 改一个边界，再解释

正常结果 13；两类 handler 调用（正常和负数），类型错误与未知工具不执行 add；内存不跨进程，stdio 跨进程。将 schema、业务拒绝和未知工具错误分别记录。将负数拒绝删掉再运行会触发断言，失败不要改成成功。

在 my_explanation 写自己的预测、观测与一个变化，导出后回学习页导入。连接正确不证明对象授权；继续用 Agent 安全专题检查可信用户边界。

In [ ]:
my_explanation=''
report={'experiment':'mcp-tool-contract','version':'2.3.0','protocol':'2026-07-28',
        'model_called':False,'scope':'official SDK; memory and stdio; no HTTP/auth',
        'checks':results[1]['checks'],'my_explanation':my_explanation}
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved mechanism_report.json')
